# 01_PINN_Basics

## Learning Goal
Introduction to Physics-Informed Neural Networks.

## Prerequisites
Basic understanding of Calculus and Neural Networks.

## Theory & Mathematics
(Mathematical derivations and fundamental concepts)

## Implementation & Experiments


In [14]:
import torch
import torch.nn as nn
import torch.autograd as autograd
import numpy as np
import matplotlib.pyplot as plt

In [15]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
torch.manual_seed(42)
np.random.seed(42)

Using device: cuda


In [16]:
class PINN(nn.Module):
    def __init__(self, layers=[2, 64, 64, 64, 64, 1]):
        super(PINN, self).__init__()
        self.activation = nn.Tanh()
        
        self.layers = nn.ModuleList()
        for i in range(len(layers)-1):
            self.layers.append(nn.Linear(layers[i], layers[i+1]))
            
        # Initialize weights using Xavier initialization
        for layer in self.layers:
            nn.init.xavier_normal_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, x, t):
        # Concatenate x and t
        z = torch.cat([x, t], dim=1)
        for i in range(len(self.layers) - 1):
            z = self.activation(self.layers[i](z))
        return self.layers[-1](z)


In [17]:
def compute_derivatives(model, x, t):
    x.requires_grad_(True)
    t.requires_grad_(True)
    
    u = model(x, t)
    
    # First derivatives (u_t, u_x)
    u_x = autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
    u_t = autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]
    
    # Second derivative (u_xx)
    u_xx = autograd.grad(u_x, x, torch.ones_like(u_x), create_graph=True)[0]
    
    return u, u_t, u_x, u_xx

## Summary
This experiment demonstrates the specific concept in action.

## Further Reading
- Scientific Machine Learning & PINN surveys.